In [1]:
import os
import librosa
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torchvision import models, transforms
from PIL import Image

In [7]:
# 1-mel_photo,把音频批处理提取Mel谱图
# 音频转Mel频谱图函数
def audio_to_mel_spectrogram(file_path, sr=22050, n_mels=128):
    y, sr = librosa.load(file_path, sr=sr)
    # y = y / np.max(np.abs(y))  # 归一化音频信号，没必要加了，Librosa默认归一化
    # print(f"Max amplitude before normalization: {np.max(np.abs(y))}")
    mel_spect = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=n_mels)
    mel_spect_db = librosa.power_to_db(mel_spect, ref=np.max)
    return mel_spect_db

# 保存频谱图为图像
def save_spectrogram_image(mel_spect_db, output_file):
    plt.figure(figsize=(10, 4))
    librosa.display.specshow(mel_spect_db, sr=22050, cmap='viridis', y_axis='mel', x_axis='time')
    plt.axis('off')
    plt.tight_layout()
    plt.savefig(output_file, bbox_inches='tight', pad_inches=0)
    plt.close()

# 批处理音频文件目录并保存频谱图
def process_audio_directory(audio_dir, output_dir):
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)
    
    audio_files = [f for f in os.listdir(audio_dir) if f.endswith('.wav')]
    
    for i, audio_file in enumerate(audio_files):
        audio_path = os.path.join(audio_dir, audio_file)
        output_image_path = os.path.join(output_dir, f'{os.path.splitext(audio_file)[0]}.png')

        mel_spect_db = audio_to_mel_spectrogram(audio_path)
        save_spectrogram_image(mel_spect_db, output_image_path)

        print(f"Saved spectrogram for {audio_file} to {output_image_path}")

# 设置目录
audio_dir = r"E:\try\data\word_single\u1"
output_dir = r"E:\data_gen\new\deepspectrum_resnet\1-mel_photo\u1"

# 执行频谱图提取
process_audio_directory(audio_dir, output_dir)


Saved spectrogram for 2023_09_18_张亚.wav to E:\data_gen\new\deepspectrum_resnet\1-mel_photo\u1\2023_09_18_张亚.png
Saved spectrogram for 2023_09_19_姜拓.wav to E:\data_gen\new\deepspectrum_resnet\1-mel_photo\u1\2023_09_19_姜拓.png
Saved spectrogram for 2023_09_19_沈习申.wav to E:\data_gen\new\deepspectrum_resnet\1-mel_photo\u1\2023_09_19_沈习申.png
Saved spectrogram for 2023_09_20_刘佑乐.wav to E:\data_gen\new\deepspectrum_resnet\1-mel_photo\u1\2023_09_20_刘佑乐.png
Saved spectrogram for 2023_09_20_尹良平.wav to E:\data_gen\new\deepspectrum_resnet\1-mel_photo\u1\2023_09_20_尹良平.png
Saved spectrogram for 2023_10_04_刘星.wav to E:\data_gen\new\deepspectrum_resnet\1-mel_photo\u1\2023_10_04_刘星.png
Saved spectrogram for 2023_10_04_史雷.wav to E:\data_gen\new\deepspectrum_resnet\1-mel_photo\u1\2023_10_04_史雷.png
Saved spectrogram for 2023_10_12_左长青.wav to E:\data_gen\new\deepspectrum_resnet\1-mel_photo\u1\2023_10_12_左长青.png
Saved spectrogram for 2023_10_16_费清友.wav to E:\data_gen\new\deepspectrum_resnet\1-mel_photo\u1\2

In [20]:
# 2-feature_npy,提取特征，保存为npy文件
# 加载预训练模型
def load_model(model_name='resnet50'):
    if model_name == 'alexnetfc6':
        # 加载预训练的 AlexNet
        model = models.alexnet(pretrained=True)
        # 截取到 FC6 层（4096维）
        model.classifier = nn.Sequential(*list(model.classifier.children())[:3])
        # print("AlexNet fc6 截取后的模型结构：", model.classifier)
    elif model_name == 'alexnet':
        # 加载预训练的 AlexNet
        model = models.alexnet(pretrained=True)
        # 截取到 FC7 层（4096维）
        model.classifier = nn.Sequential(*list(model.classifier.children())[:6])
        # print("AlexNet fc7 截取后的模型结构：", model.classifier)
    elif model_name == 'resnet50':
        # 加载预训练的 ResNet50
        model = models.resnet50(pretrained=True)
        # 截取到全局平均池化层（2048维）
        model = nn.Sequential(*list(model.children())[:-2], nn.AdaptiveAvgPool2d((1, 1)), nn.Flatten())
    elif model_name == 'densenet121':
        # 加载预训练的 DenseNet121
        model = models.densenet121(pretrained=True)
        # 截取到 DenseNet 的 feature 层（1024维）
        model = nn.Sequential(model.features, nn.ReLU(), nn.AdaptiveAvgPool2d((1, 1)), nn.Flatten())
    else:
        raise ValueError(f"Unsupported model: {model_name}")
    
    # 切换到评估模式
    model.eval()
    return model

# 图像预处理
preprocess = transforms.Compose([
    transforms.Resize(224),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

# 提取Deep Spectrum特征
def extract_deep_spectrum_features(image_path, model):
    # 加载图像并预处理
    img = Image.open(image_path).convert('RGB')
    img_tensor = preprocess(img).unsqueeze(0)  # 添加 batch 维度
    
    # 提取特征
    with torch.no_grad():
        features = model(img_tensor)
    
    # 展平特征向量并返回
    return features.squeeze().numpy()

# 批处理频谱图并保存特征
def process_spectrogram_directory(spectrogram_dir, feature_save_dir, model_name='resnet50'):
    # 加载指定模型
    model = load_model(model_name)
    feature_list = []
    spectrogram_files = [f for f in os.listdir(spectrogram_dir) if f.endswith('.png')]

    for i, spectrogram_file in enumerate(spectrogram_files):
        image_path = os.path.join(spectrogram_dir, spectrogram_file)
        features = extract_deep_spectrum_features(image_path, model)
        feature_list.append(features)
        print(f"Processed {i + 1}/{len(spectrogram_files)}: {spectrogram_file} - Feature shape: {features.shape}")

    # 转换为 NumPy 数组并保存
    features = np.array(feature_list)
    feature_file = os.path.join(feature_save_dir, f'u1_{model_name}.npy')
    np.save(feature_file, features)
    print(f"特征提取完成，保存为 '{feature_file}'")

    # 预览特征
    loaded_features = np.load(feature_file)
    print("预览特征数组形状:", loaded_features.shape)
    print("特征数组的前几项:", loaded_features[:5])  # 预览前5项特征

# 设置频谱图目录和保存路径
spectrogram_dir = r"E:\data_gen\new\deepspectrum_resnet\1-mel_photo\u1"
feature_save_dir = r"E:\data_gen\new\deepspectrum_resnet\2-feature_npy"

# 提取特征并保存
process_spectrogram_directory(spectrogram_dir, feature_save_dir, model_name='alexnet')  # 可切换为'alexnet'，'resnet50' 或 'densenet121'

Processed 1/113: 2023_09_18_张亚.png - Feature shape: (4096,)
Processed 2/113: 2023_09_19_姜拓.png - Feature shape: (4096,)
Processed 3/113: 2023_09_19_沈习申.png - Feature shape: (4096,)
Processed 4/113: 2023_09_20_刘佑乐.png - Feature shape: (4096,)
Processed 5/113: 2023_09_20_尹良平.png - Feature shape: (4096,)
Processed 6/113: 2023_10_04_刘星.png - Feature shape: (4096,)
Processed 7/113: 2023_10_04_史雷.png - Feature shape: (4096,)
Processed 8/113: 2023_10_12_左长青.png - Feature shape: (4096,)
Processed 9/113: 2023_10_16_费清友.png - Feature shape: (4096,)
Processed 10/113: 2023_10_16_陈伟.png - Feature shape: (4096,)
Processed 11/113: 2023_10_17_左万银.png - Feature shape: (4096,)
Processed 12/113: 2023_10_20_杨兆和.png - Feature shape: (4096,)
Processed 13/113: 2023_10_27_王义杰.png - Feature shape: (4096,)
Processed 14/113: 2023_10_27_胡伟.png - Feature shape: (4096,)
Processed 15/113: 2023_10_29_徐淮.png - Feature shape: (4096,)
Processed 16/113: 2023_10_31_王海洲.png - Feature shape: (4096,)
Processed 17/113: 2023_1

In [21]:
# 3-label_xlsx，匹配标签，生成xlsx文件
# 文件路径设置
npy_folder = r"E:\data_gen\new\deepspectrum_resnet\2-feature_npy"  # npy文件夹路径
labels_file = r"E:\try\音视频+ahi+淮安.xlsx"  # 标签文件路径
sheet_name = '标签'  # 标签表的Sheet名称
label_range = 'F'  # 标签区域
output_dir = r"E:\data_gen\new\deepspectrum_resnet\3-label_xlsx\ahi_30"  # 输出文件夹路径

# 创建输出目录
if not os.path.exists(output_dir):
    os.makedirs(output_dir)

# 读取标签
labels_df = pd.read_excel(labels_file, sheet_name=sheet_name, usecols=label_range)
labels = labels_df.iloc[:, 0].values

# 遍历文件夹中的所有npy文件
for npy_file in os.listdir(npy_folder):
    if npy_file.endswith('.npy'):
        npy_path = os.path.join(npy_folder, npy_file)

        # 加载npy文件中的特征
        features = np.load(npy_path)
        print(f"处理文件: {npy_file}, 样本数量: {features.shape[0]}")
        
        # 检查特征数量和标签数量是否匹配
        if len(labels) != features.shape[0]:
            raise ValueError(f"样本数量与标签数量不匹配! 文件: {npy_file}")
        
        # 将特征转换为DataFrame并添加标签
        combined_df = pd.DataFrame(features)
        combined_df['label'] = labels

        # 生成文件名并保存为xlsx文件
        output_excel_file = os.path.join(output_dir, f"{os.path.splitext(npy_file)[0]}_label.xlsx")
        combined_df.to_excel(output_excel_file, index=False)
        print(f"保存完成: {output_excel_file}")

print("所有文件处理完成。")

处理文件: a1_alexnet.npy, 样本数量: 113
保存完成: E:\data_gen\new\deepspectrum_resnet\3-label_xlsx\ahi_30\a1_alexnet_label.xlsx
处理文件: a1_densenet121.npy, 样本数量: 113
保存完成: E:\data_gen\new\deepspectrum_resnet\3-label_xlsx\ahi_30\a1_densenet121_label.xlsx
处理文件: a1_resnet50.npy, 样本数量: 113
保存完成: E:\data_gen\new\deepspectrum_resnet\3-label_xlsx\ahi_30\a1_resnet50_label.xlsx
处理文件: e1_alexnet.npy, 样本数量: 113
保存完成: E:\data_gen\new\deepspectrum_resnet\3-label_xlsx\ahi_30\e1_alexnet_label.xlsx
处理文件: e1_densenet121.npy, 样本数量: 113
保存完成: E:\data_gen\new\deepspectrum_resnet\3-label_xlsx\ahi_30\e1_densenet121_label.xlsx
处理文件: e1_resnet50.npy, 样本数量: 113
保存完成: E:\data_gen\new\deepspectrum_resnet\3-label_xlsx\ahi_30\e1_resnet50_label.xlsx
处理文件: i1_alexnet.npy, 样本数量: 113
保存完成: E:\data_gen\new\deepspectrum_resnet\3-label_xlsx\ahi_30\i1_alexnet_label.xlsx
处理文件: i1_densenet121.npy, 样本数量: 113
保存完成: E:\data_gen\new\deepspectrum_resnet\3-label_xlsx\ahi_30\i1_densenet121_label.xlsx
处理文件: i1_resnet50.npy, 样本数量: 113
保存完成: E:\da